In [1]:
import pandas as pd
from pathlib import Path

In [9]:
# Input files
file_C = "../results/Fluconazole_C_genes_20260312.csv"   # Control
file_F = "../results/Fluconazole_F_genes_20260312.csv"   # 1xMIC

df_C = pd.read_csv(file_C)
df_F = pd.read_csv(file_F)

print("C shape:", df_C.shape)
print("F shape:", df_F.shape)

print("\nColumns in C:", df_C.columns.tolist())
print("Columns in F:", df_F.columns.tolist())

C shape: (87, 9)
F shape: (45, 9)

Columns in C: ['interval_id', 'chrom', 'start', 'end', 'genes', 'genes_std', 'pleio_score', 'pleio_percentile', 'lod_max']
Columns in F: ['interval_id', 'chrom', 'start', 'end', 'genes', 'genes_std', 'pleio_score', 'pleio_percentile', 'lod_max']


In [10]:
def split_genes(val):
    """
    Split a comma-separated genes_std entry into a clean set of genes.
    """
    if pd.isna(val):
        return set()
    return {g.strip() for g in str(val).split(",") if g.strip()}

def extract_gene_set(df, column="genes_std"):
    """
    Extract all unique genes across a dataframe column.
    """
    all_genes = set()
    for val in df[column]:
        all_genes |= split_genes(val)
    return all_genes

def filter_intervals_by_gene_set(df, gene_set, gene_column="genes_std"):
    """
    Keep interval rows where at least one gene in gene_column belongs to gene_set.
    Adds a column 'matched_genes' showing which genes matched.
    """
    out = df.copy()
    out["matched_genes"] = out[gene_column].apply(
        lambda x: ",".join(sorted(split_genes(x) & gene_set))
    )
    out = out[out["matched_genes"] != ""].copy()
    return out

In [11]:
genes_C = extract_gene_set(df_C, column="genes_std")
genes_F = extract_gene_set(df_F, column="genes_std")

overlap_genes = genes_C & genes_F
C_only_genes = genes_C - genes_F
F_only_genes = genes_F - genes_C
non_overlap_genes = genes_C ^ genes_F   # symmetric difference

print(f"Unique genes in C:          {len(genes_C)}")
print(f"Unique genes in F:          {len(genes_F)}")
print(f"Overlapping genes:          {len(overlap_genes)}")
print(f"C-only genes:               {len(C_only_genes)}")
print(f"F-only genes:               {len(F_only_genes)}")
print(f"All non-overlapping genes:  {len(non_overlap_genes)}")

Unique genes in C:          770
Unique genes in F:          330
Overlapping genes:          91
C-only genes:               679
F-only genes:               239
All non-overlapping genes:  918


In [12]:
# Intervals in C containing overlapping genes
df_C_overlap_intervals = filter_intervals_by_gene_set(df_C, overlap_genes)

# Intervals in F containing overlapping genes
df_F_overlap_intervals = filter_intervals_by_gene_set(df_F, overlap_genes)

# Intervals in C containing C-only genes
df_C_only_intervals = filter_intervals_by_gene_set(df_C, C_only_genes)

# Intervals in F containing F-only genes
df_F_only_intervals = filter_intervals_by_gene_set(df_F, F_only_genes)

# Intervals in C containing any non-overlapping gene
df_C_nonoverlap_intervals = filter_intervals_by_gene_set(df_C, non_overlap_genes)

# Intervals in F containing any non-overlapping gene
df_F_nonoverlap_intervals = filter_intervals_by_gene_set(df_F, non_overlap_genes)

print("C intervals with overlapping genes:     ", df_C_overlap_intervals.shape[0])
print("F intervals with overlapping genes:     ", df_F_overlap_intervals.shape[0])
print("C intervals with C-only genes:          ", df_C_only_intervals.shape[0])
print("F intervals with F-only genes:          ", df_F_only_intervals.shape[0])
print("C intervals with non-overlapping genes: ", df_C_nonoverlap_intervals.shape[0])
print("F intervals with non-overlapping genes: ", df_F_nonoverlap_intervals.shape[0])

C intervals with overlapping genes:      17
F intervals with overlapping genes:      22
C intervals with C-only genes:           84
F intervals with F-only genes:           36
C intervals with non-overlapping genes:  84
F intervals with non-overlapping genes:  36


In [13]:
keep_cols = [
    "interval_id",
    "chrom",
    "start",
    "end",
    "genes",
    "genes_std",
    "pleio_score",
    "pleio_percentile",
    "lod_max",
    "matched_genes"
]

df_C_overlap_intervals = df_C_overlap_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])
df_F_overlap_intervals = df_F_overlap_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])

df_C_only_intervals = df_C_only_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])
df_F_only_intervals = df_F_only_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])

df_C_nonoverlap_intervals = df_C_nonoverlap_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])
df_F_nonoverlap_intervals = df_F_nonoverlap_intervals[keep_cols].sort_values(["chrom", "start", "end", "interval_id"])

In [14]:
print("C intervals containing non-overlapping genes:")
display(df_C_nonoverlap_intervals.head(10))

print("F intervals containing non-overlapping genes:")
display(df_F_nonoverlap_intervals.head(10))

C intervals containing non-overlapping genes:


,interval_id,chrom,start,end,genes,genes_std,pleio_score,pleio_percentile,lod_max,matched_genes
20,3841,2,303528,603378,"YBR033W, YBR036C, YBR037C, YBR038W, YBR039W, Y...","EDS1, CSG2, SCO1, CHS2, ATP3, FAT1, CST26, QDR...",0.063315,99.667908,14.262910,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,AMN1,APD1,..."
21,3883,2,303528,603378,"YBR033W, YBR036C, YBR037C, YBR038W, YBR039W, Y...","EDS1, CSG2, SCO1, CHS2, ATP3, FAT1, CST26, QDR...",0.063315,99.667908,14.329451,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,AMN1,APD1,..."
18,3569,2,334070,552398,"YBR048W, YBR049C, YBR050C, YBR052C, YBR053C, Y...","RPS11B, REB1, REG2, RFS1, YBR053C, YRO2, PRP6,...",0.063315,99.667908,24.236473,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,APD1,ARA1,..."
19,3570,2,334070,552398,"YBR048W, YBR049C, YBR050C, YBR052C, YBR053C, Y...","RPS11B, REB1, REG2, RFS1, YBR053C, YRO2, PRP6,...",0.063315,99.667908,24.236473,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,APD1,ARA1,..."
0,2203,2,337611,552050,"YBR049C, YBR050C, YBR052C, YBR053C, YBR054W, Y...","REB1, REG2, RFS1, YBR053C, YRO2, PRP6, MRX18, ...",0.063315,99.667908,24.818703,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,APD1,ARA1,..."
17,3484,2,346663,543147,"YBR055C, YBR056W, YBR056W-A, YBR057C, YBR058C,...","PRP6, MRX18, MNC1, MUM2, UBP14, AKL1, ORC2, TR...",0.063315,99.667908,25.948817,"AAC3,ADH5,AGP2,AIM3,AKL1,ALG1,ALG14,ARA1,ATG14..."
15,3318,2,355836,529278,"YBR058C, YBR059C, YBR060C, YBR061C, YBR062C, Y...","UBP14, AKL1, ORC2, TRM7, YBR062C, CNM1, CNM1, ...",0.061356,99.619701,30.024351,"AAC3,AGP2,AIM3,AKL1,ALG1,ALG14,ATG14,ATG42,BAP..."
16,3319,2,355836,529278,"YBR058C, YBR059C, YBR060C, YBR061C, YBR062C, Y...","UBP14, AKL1, ORC2, TRM7, YBR062C, CNM1, CNM1, ...",0.061356,99.619701,30.024351,"AAC3,AGP2,AIM3,AKL1,ALG1,ALG14,ATG14,ATG42,BAP..."
1,2329,2,361468,524543,"YBR060C, YBR061C, YBR062C, YBR063C, YBR064W, Y...","ORC2, TRM7, YBR062C, CNM1, CNM1, ECM2, NRG2, T...",0.061356,99.619701,30.668833,"AAC3,AGP2,AIM3,ALG1,ALG14,ATG14,ATG42,BAP2,CBP..."
2,2330,2,361468,524543,"YBR060C, YBR061C, YBR062C, YBR063C, YBR064W, Y...","ORC2, TRM7, YBR062C, CNM1, CNM1, ECM2, NRG2, T...",0.061356,99.619701,30.668833,"AAC3,AGP2,AIM3,ALG1,ALG14,ATG14,ATG42,BAP2,CBP..."


F intervals containing non-overlapping genes:


,interval_id,chrom,start,end,genes,genes_std,pleio_score,pleio_percentile,lod_max,matched_genes
1,268,1,46030,53935,"YAL053W, YAL051W, YAL049C, YAL048C, YAL047W-A","FLC2, OAF1, AIM2, GEM1, GEM1",0.035920,98.185543,39.443184,"AIM2,FLC2,GEM1,OAF1"
4,10615,5,339948,434766,"YER091C, YER091C-A, YER093C, YER093C-A, YER094...","MET6, IES5, TSC11, AIM11, PUP3, SHC1, YNCE0018...",0.075441,99.783069,14.248954,"AIM11,AST2,AVT6,BOI2,COM2,DSE1,GLC7,GLE2,GLO3,..."
5,10616,5,339948,434766,"YER091C, YER091C-A, YER093C, YER093C-A, YER094...","MET6, IES5, TSC11, AIM11, PUP3, SHC1, YNCE0018...",0.075441,99.783069,14.248954,"AIM11,AST2,AVT6,BOI2,COM2,DSE1,GLC7,GLE2,GLO3,..."
3,10486,5,372639,389528,"YER105C, YER106W, YER107C, YER107W-A, YER110C,...","NUP157, MAM1, GLE2, GLE2, KAP123, SWI4, LSM4, ...",0.075441,99.783069,21.274149,"GLE2,KAP123,LSM4,MAM1,NUP157,SWI4,TMN3"
6,13535,7,159246,193938,"YGL181W, YGL180W, YGL179C, YGL178W, YGL176C, Y...","GTS1, ATG1, TOS3, MPT5, YGL176C, SAE2, BUD13, ...",0.024765,94.416026,15.001065,"ATG1,BUD13,CUP2,GTS1,HUR1,MPT5,NUP49,PMR1,RAD5..."
7,13645,7,159246,193938,"YGL181W, YGL180W, YGL179C, YGL178W, YGL176C, Y...","GTS1, ATG1, TOS3, MPT5, YGL176C, SAE2, BUD13, ...",0.024765,94.416026,15.284698,"ATG1,BUD13,CUP2,GTS1,HUR1,MPT5,NUP49,PMR1,RAD5..."
9,14606,7,453554,515594,"YGL022W, YGL021W, YGL020C, YGL019W, YGL018C, Y...","STT3, ALK1, GET1, CKB1, JAC1, ATE1, KAP122, BI...",0.039901,98.816251,98.888231,"ALK1,ATE1,BIL2,BRP1,CDH1,CKB1,COG7,CUL3,ECT1,E..."
10,14607,7,453554,515594,"YGL022W, YGL021W, YGL020C, YGL019W, YGL018C, Y...","STT3, ALK1, GET1, CKB1, JAC1, ATE1, KAP122, BI...",0.039901,98.816251,98.888231,"ALK1,ATE1,BIL2,BRP1,CDH1,CKB1,COG7,CUL3,ECT1,E..."
8,14439,7,470529,473502,"YGL013C, YGL012W, YGL011C","PDR1, ERG4, SCL1",0.015480,80.410563,116.945234,"ERG4,PDR1,SCL1"
11,15200,7,806826,829946,"YGR158C, YGR159C, YGR160W, YGR161C, YGR161W-C,...","MTR3, NSR1, NSR1, RTS3, YGR161W-C, TIF4631, MR...",0.023203,93.323335,12.079738,"MRPS35,MTR3,NSR1,RTS3,TIF4631,TRS65,YGR161W-C"


In [15]:
# Non-overlapping interval outputs
df_C_nonoverlap_intervals.to_csv("Fluconazole_C_nonoverlap_gene_intervals.csv", index=False)
df_F_nonoverlap_intervals.to_csv("Fluconazole_F_nonoverlap_gene_intervals.csv", index=False)

# Overlapping interval outputs
df_C_overlap_intervals.to_csv("Fluconazole_C_overlap_gene_intervals.csv", index=False)
df_F_overlap_intervals.to_csv("Fluconazole_F_overlap_gene_intervals.csv", index=False)

# Side-specific unique interval outputs
df_C_only_intervals.to_csv("Fluconazole_C_only_gene_intervals.csv", index=False)
df_F_only_intervals.to_csv("Fluconazole_F_only_gene_intervals.csv", index=False)

print("Saved files:")
# print(" - Fluconazole_C_nonoverlap_gene_intervals.csv")
print(" - Fluconazole_F_nonoverlap_gene_intervals.csv")
# print(" - Fluconazole_C_overlap_gene_intervals.csv")
# print(" - Fluconazole_F_overlap_gene_intervals.csv")
# print(" - Fluconazole_C_only_gene_intervals.csv")
print(" - Fluconazole_F_only_gene_intervals.csv")

Saved files:
 - Fluconazole_F_nonoverlap_gene_intervals.csv
 - Fluconazole_F_only_gene_intervals.csv


In [16]:
# Confirm no matched gene appears in Control
assert all(
    gene not in genes_C
    for row in df_F_only_intervals["matched_genes"]
    for gene in row.split(",")
)